# Flux-capable.flx Coherency Test

**Testing the model created by Phase 9.7 MASSIVE Dataset Injection**

This notebook evaluates:
1. **Generation coherence** - Does output make sense?
2. **Instruction following** - Does it follow prompts?
3. **Domain coverage** - Knowledge across injected domains
4. **Comparison** - vs Flux-X-complete.flx baseline

---

In [ ]:
# Cell 1: Setup
import os
from pathlib import Path

REPO_URL = 'https://github.com/Unseengap/FLUX.git'
ROOT = Path('/kaggle/working/FLUX') if Path('/kaggle').exists() else Path('/content/FLUX')

if ROOT.exists():
    print('  Pulling latest...')
    !cd {ROOT} && git pull
else:
    print('  Cloning repo...')
    !git clone {REPO_URL} {ROOT}

os.chdir(ROOT)
!pip install -q -e . 2>/dev/null || pip install -q -r requirements.txt
!pip install -q huggingface_hub
print(f'  ✓ Working dir: {os.getcwd()}')

In [ ]:
# Cell 2: Paths & Imports
import sys
import torch
from pathlib import Path

ROOT = Path('/kaggle/working/FLUX') if Path('/kaggle').exists() else Path('/content/FLUX')
PHASES_DIR = ROOT / 'phases'
CHECKPOINT_DIR = ROOT / 'checkpoints'

# Add phase paths
for subdir in ['phase8_8', 'phase8_9', 'phase9']:
    p = PHASES_DIR / subdir
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from flux_utils import get_device
device = get_device()
print(f'  Device: {device}')

if device == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Cell 3: Locate Models (Local First, then HuggingFace)
import os
from huggingface_hub import hf_hub_download

print("  MODEL TO TEST: Flux-capable.flx (Phase 9.7 injection)")
print("  BASELINE: Flux-X-complete.flx (original)")
print("  " + "="*50)

# Get HF token for fallback download
hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
except:
    pass
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
    except:
        pass
if not hf_token:
    hf_token = os.environ.get('HF_TOKEN')

# ─────────────────────────────────────────────
# 1. Flux-capable.flx (THE MODEL WE'RE TESTING)
# ─────────────────────────────────────────────
capable_path = CHECKPOINT_DIR / 'Flux-capable.flx'

if capable_path.exists():
    size_mb = capable_path.stat().st_size / 1e6
    size_gb = size_mb / 1000
    print(f'\n  ✓ FOUND LOCAL: Flux-capable.flx ({size_gb:.2f} GB)')
    if size_mb < 500:
        print(f'    ⚠ WARNING: File seems small - expected 2-6 GB')
else:
    print('\n  Flux-capable.flx not found locally, downloading from HuggingFace...')
    try:
        hf_hub_download(
            repo_id='UnseenGAP/FLUX',
            filename='checkpoints/Flux-capable.flx',
            local_dir=str(ROOT),
            token=hf_token,
        )
        print(f'  ✓ Downloaded Flux-capable.flx ({capable_path.stat().st_size/1e6:.1f} MB)')
    except Exception as e:
        raise FileNotFoundError(
            f"Flux-capable.flx not found!\n"
            f"Either run Phase 9.7 notebook first, or upload to HuggingFace.\n"
            f"Error: {e}"
        )

# ─────────────────────────────────────────────
# 2. Flux-X-complete.flx (BASELINE FOR COMPARISON)
# ─────────────────────────────────────────────
baseline_path = CHECKPOINT_DIR / 'Flux-X-complete.flx'

if baseline_path.exists():
    print(f'  ✓ FOUND LOCAL: Flux-X-complete.flx ({baseline_path.stat().st_size/1e6:.1f} MB)')
else:
    print('  Downloading Flux-X-complete.flx (baseline)...')
    hf_hub_download(
        repo_id='UnseenGAP/FLUX',
        filename='checkpoints/Flux-X-complete.flx',
        local_dir=str(ROOT),
        token=hf_token,
    )
    print(f'  ✓ Downloaded Flux-X-complete.flx ({baseline_path.stat().st_size/1e6:.1f} MB)')

print("\n  " + "="*50)
print("  Ready to test Flux-capable.flx")

In [ ]:
# Cell 4: Load Flux-capable Model
from flx_loader import load_flux_from_flx, get_flx_info

print('\n  Loading Flux-capable.flx...')
info = get_flx_info(capable_path)
print(f'  Version: {info["version"]}')
print(f'  Metadata: {info.get("metadata", {}).get("description", "N/A")[:60]}...')

model = load_flux_from_flx(
    path=capable_path,
    device=device,
    verbose=True,
)

stats = model.get_stats()
print(f'\n  ✓ Model loaded')
print(f'    Params: {stats.total_params:,}')
print(f'    Field energy: {stats.field_energy:.4f}')

---
## Test 1: Basic Generation Coherence

Simple prompts to test if output is coherent English.

In [ ]:
# Cell 5: Basic Generation Test
print('\n' + '='*60)
print('  TEST 1: BASIC GENERATION COHERENCE')
print('='*60)

basic_prompts = [
    "Hello, how are you today?",
    "The weather is",
    "Once upon a time",
    "The capital of France is",
    "Water boils at",
]

model.eval()
with torch.no_grad():
    for prompt in basic_prompts:
        try:
            response = model.generate(prompt, max_length=50, temperature=0.7)
            text = response.text if hasattr(response, 'text') else str(response)
            # Trim to reasonable length for display
            text = text[:100] + '...' if len(text) > 100 else text
            print(f'\n  PROMPT: "{prompt}"')
            print(f'  OUTPUT: {text}')
        except Exception as e:
            print(f'\n  PROMPT: "{prompt}"')
            print(f'  ERROR: {e}')

---
## Test 2: Instruction Following

Test if the model follows instructions (SlimOrca, OpenHermes domains).

In [ ]:
# Cell 6: Instruction Following Test
print('\n' + '='*60)
print('  TEST 2: INSTRUCTION FOLLOWING')
print('='*60)

instruction_prompts = [
    "User: Explain photosynthesis in simple terms.\nAssistant:",
    "User: Write a haiku about coding.\nAssistant:",
    "User: List 3 benefits of exercise.\nAssistant:",
    "User: What is machine learning?\nAssistant:",
    "User: Translate 'hello' to Spanish.\nAssistant:",
]

with torch.no_grad():
    for prompt in instruction_prompts:
        try:
            response = model.generate(prompt, max_length=80, temperature=0.7)
            text = response.text if hasattr(response, 'text') else str(response)
            # Extract just the response part
            if 'Assistant:' in text:
                text = text.split('Assistant:')[-1].strip()
            text = text[:150] + '...' if len(text) > 150 else text
            
            # Show just the question
            question = prompt.split('\n')[0].replace('User: ', '')
            print(f'\n  Q: {question}')
            print(f'  A: {text}')
        except Exception as e:
            print(f'\n  Q: {prompt[:40]}...')
            print(f'  ERROR: {e}')

---
## Test 3: Math & Reasoning (GSM8K, MathInstruct domains)

In [ ]:
# Cell 7: Math & Reasoning Test
print('\n' + '='*60)
print('  TEST 3: MATH & REASONING')
print('='*60)

math_prompts = [
    "Problem: If I have 5 apples and give away 2, how many do I have?\n\nSolution:",
    "Problem: What is 15% of 80?\n\nSolution:",
    "Problem: A train travels 60 miles in 1 hour. How far in 2.5 hours?\n\nSolution:",
    "Problem: Simplify: 2x + 3x - x\n\nSolution:",
]

with torch.no_grad():
    for prompt in math_prompts:
        try:
            response = model.generate(prompt, max_length=100, temperature=0.5)
            text = response.text if hasattr(response, 'text') else str(response)
            if 'Solution:' in text:
                text = text.split('Solution:')[-1].strip()
            text = text[:200] + '...' if len(text) > 200 else text
            
            problem = prompt.split('\n')[0].replace('Problem: ', '')
            print(f'\n  Problem: {problem}')
            print(f'  Solution: {text}')
        except Exception as e:
            print(f'\n  Problem: {prompt[:40]}...')
            print(f'  ERROR: {e}')

---
## Test 4: Code Generation (CodeAlpaca domain)

In [ ]:
# Cell 8: Code Generation Test
print('\n' + '='*60)
print('  TEST 4: CODE GENERATION')
print('='*60)

code_prompts = [
    "def fibonacci(n):",
    "# Function to check if a number is prime\ndef is_prime(n):",
    "# Python function to reverse a string\ndef reverse_string(s):",
    "Task: Write a function to find the maximum element in a list.\nCode:",
]

with torch.no_grad():
    for prompt in code_prompts:
        try:
            response = model.generate(prompt, max_length=120, temperature=0.6)
            text = response.text if hasattr(response, 'text') else str(response)
            text = text[:250] + '...' if len(text) > 250 else text
            
            print(f'\n  PROMPT: {prompt[:50]}...' if len(prompt) > 50 else f'\n  PROMPT: {prompt}')
            print(f'  OUTPUT:')
            for line in text.split('\n')[:8]:
                print(f'    {line}')
        except Exception as e:
            print(f'\n  PROMPT: {prompt[:40]}...')
            print(f'  ERROR: {e}')

---
## Test 5: Conversation / Multi-turn (Capybara, UltraChat domains)

In [ ]:
# Cell 9: Conversation Test
print('\n' + '='*60)
print('  TEST 5: CONVERSATION / MULTI-TURN')
print('='*60)

conversation_prompts = [
    "user: Hi there!\nassistant: Hello! How can I help you today?\nuser: Can you tell me about climate change?\nassistant:",
    "Human: What are the main programming paradigms?\nAssistant: The main programming paradigms are...\nHuman: Can you explain object-oriented programming?\nAssistant:",
]

with torch.no_grad():
    for prompt in conversation_prompts:
        try:
            response = model.generate(prompt, max_length=100, temperature=0.7)
            text = response.text if hasattr(response, 'text') else str(response)
            # Get just the last assistant response
            if 'assistant:' in text.lower() or 'Assistant:' in text:
                parts = text.lower().split('assistant:')
                text = parts[-1].strip() if parts else text
            text = text[:200] + '...' if len(text) > 200 else text
            
            # Show context
            context_lines = prompt.split('\n')[-2:]
            print(f'\n  CONTEXT: ...{context_lines[0][:50]}')
            print(f'  RESPONSE: {text}')
        except Exception as e:
            print(f'\n  CONTEXT: {prompt[:40]}...')
            print(f'  ERROR: {e}')

---
## Test 6: Comparison with Baseline (Flux-X-complete.flx)

Load baseline model and compare outputs on same prompts.

In [ ]:
# Cell 10: Load Baseline Model
import gc

print('\n  Loading baseline model for comparison...')

# Clear capable model from GPU
model.cpu()
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Load baseline
baseline_model = load_flux_from_flx(
    path=baseline_path,
    device=device,
    verbose=False,
)

print('  ✓ Baseline model loaded')

In [ ]:
# Cell 11: Side-by-Side Comparison
print('\n' + '='*60)
print('  TEST 6: SIDE-BY-SIDE COMPARISON')
print('='*60)

comparison_prompts = [
    "User: What is the capital of Japan?\nAssistant:",
    "Problem: What is 7 × 8?\nSolution:",
    "def hello_world():",
]

# Move capable model back to GPU/device
model.to(device)

with torch.no_grad():
    for prompt in comparison_prompts:
        print(f'\n  PROMPT: "{prompt[:50]}..."' if len(prompt) > 50 else f'\n  PROMPT: "{prompt}"')
        print('  ' + '-'*50)
        
        # Capable model
        try:
            capable_resp = model.generate(prompt, max_length=60, temperature=0.7)
            capable_text = capable_resp.text if hasattr(capable_resp, 'text') else str(capable_resp)
            capable_text = capable_text[:100]
        except Exception as e:
            capable_text = f'ERROR: {e}'
        
        # Baseline model
        try:
            baseline_resp = baseline_model.generate(prompt, max_length=60, temperature=0.7)
            baseline_text = baseline_resp.text if hasattr(baseline_resp, 'text') else str(baseline_resp)
            baseline_text = baseline_text[:100]
        except Exception as e:
            baseline_text = f'ERROR: {e}'
        
        print(f'  [CAPABLE]:  {capable_text}')
        print(f'  [BASELINE]: {baseline_text}')

---
## Test 7: Coherence Metrics

Quantitative analysis of generation quality.

In [ ]:
# Cell 12: Coherence Metrics
import re
from collections import Counter

print('\n' + '='*60)
print('  TEST 7: COHERENCE METRICS')
print('='*60)

test_prompts = [
    "The quick brown fox",
    "In the beginning",
    "User: Hello\nAssistant:",
    "def add(a, b):",
    "The year 2024",
] * 3  # 15 total samples

def analyze_text(text):
    """Compute simple coherence metrics."""
    words = re.findall(r'\b\w+\b', text.lower())
    
    if len(words) == 0:
        return {'words': 0, 'unique_ratio': 0, 'avg_word_len': 0, 'has_repetition': True}
    
    unique = len(set(words))
    unique_ratio = unique / len(words)
    avg_word_len = sum(len(w) for w in words) / len(words)
    
    # Check for excessive repetition (same word > 3 times in a row)
    has_repetition = False
    for i in range(len(words) - 3):
        if words[i] == words[i+1] == words[i+2] == words[i+3]:
            has_repetition = True
            break
    
    return {
        'words': len(words),
        'unique_ratio': unique_ratio,
        'avg_word_len': avg_word_len,
        'has_repetition': has_repetition,
    }

metrics = []
with torch.no_grad():
    for prompt in test_prompts:
        try:
            response = model.generate(prompt, max_length=60, temperature=0.7)
            text = response.text if hasattr(response, 'text') else str(response)
            m = analyze_text(text)
            metrics.append(m)
        except:
            metrics.append({'words': 0, 'unique_ratio': 0, 'avg_word_len': 0, 'has_repetition': True})

# Aggregate
avg_words = sum(m['words'] for m in metrics) / len(metrics)
avg_unique = sum(m['unique_ratio'] for m in metrics) / len(metrics)
avg_word_len = sum(m['avg_word_len'] for m in metrics) / len(metrics)
repetition_rate = sum(1 for m in metrics if m['has_repetition']) / len(metrics)

print(f'\n  Samples analyzed: {len(metrics)}')
print(f'  Average words per response: {avg_words:.1f}')
print(f'  Vocabulary diversity: {avg_unique:.2f} (unique/total ratio)')
print(f'  Average word length: {avg_word_len:.1f} chars')
print(f'  Repetition rate: {repetition_rate*100:.1f}%')

# Simple score
coherence_score = (avg_unique * 50) + (min(avg_words, 30) / 30 * 30) + ((1 - repetition_rate) * 20)
print(f'\n  COHERENCE SCORE: {coherence_score:.1f} / 100')

if coherence_score >= 70:
    print('  ✓ GOOD: Model generates coherent, varied text')
elif coherence_score >= 50:
    print('  ⚠ MODERATE: Some coherence issues')
else:
    print('  ✗ POOR: Significant coherence problems')

---
## Summary

In [ ]:
# Cell 13: Summary
print("""
╔════════════════════════════════════════════════════════════════════╗
║              FLUX-CAPABLE COHERENCY TEST COMPLETE                 ║
╠════════════════════════════════════════════════════════════════════╣
║                                                                    ║
║  TESTS RUN:                                                        ║
║  ├── Basic Generation Coherence                                   ║
║  ├── Instruction Following                                        ║
║  ├── Math & Reasoning                                             ║
║  ├── Code Generation                                              ║
║  ├── Conversation / Multi-turn                                    ║
║  ├── Side-by-Side Comparison with Baseline                        ║
║  └── Quantitative Coherence Metrics                               ║
║                                                                    ║
║  MODEL TESTED: Flux-capable.flx                                    ║
║  BASELINE: Flux-X-complete.flx                                     ║
║                                                                    ║
╚════════════════════════════════════════════════════════════════════╝
""")

print(f'  Coherence Score: {coherence_score:.1f}/100')
print(f'  Vocabulary Diversity: {avg_unique:.2f}')
print(f'  Repetition Rate: {repetition_rate*100:.1f}%')